In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("../data/train.csv")

In [4]:
df.shape

(246008, 122)

In [5]:
df.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,310536,0,Cash loans,F,N,N,2,90000.0,227520.0,13189.5,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0
1,365516,0,Cash loans,M,Y,Y,0,90000.0,161730.0,13095.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,242055,1,Cash loans,M,N,Y,0,135000.0,728847.0,26307.0,...,0,0,0,0,0.0,0.0,2.0,0.0,0.0,2.0
3,454894,1,Cash loans,M,N,N,0,135000.0,474183.0,34636.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,4.0
4,448321,0,Cash loans,F,N,Y,0,180000.0,254700.0,27558.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
#check for duplicate values
df.duplicated().sum()

np.int64(0)

In [7]:
#check for duplicate values in the id column
df['SK_ID_CURR'].nunique() == df.shape[0]

True

In [8]:
df["TARGET"].value_counts()/df.shape[0]

TARGET
0    0.919271
1    0.080729
Name: count, dtype: float64

In [9]:
#summary table
cols = df.columns
sub = df[cols]

summary = pd.DataFrame({
  "col_num": range(len(cols)),
  "column": cols,
  "dtype": sub.dtypes.values,
  "n_unique": sub.nunique().values,
  "%_missing": sub.isna().sum().values/df.shape[0]*100
})

summary.set_index("col_num")

with pd.option_context("display.max_rows", None):
    #display(summary.sort_values('%_missing', ascending=False))
    display(summary)


,col_num,column,dtype,n_unique,%_missing
0,0,SK_ID_CURR,int64,246008,0.000000
1,1,TARGET,int64,2,0.000000
2,2,NAME_CONTRACT_TYPE,str,2,0.000000
3,3,CODE_GENDER,str,3,0.000000
4,4,FLAG_OWN_CAR,str,2,0.000000
5,5,FLAG_OWN_REALTY,str,2,0.000000
6,6,CNT_CHILDREN,int64,15,0.000000
7,7,AMT_INCOME_TOTAL,float64,2168,0.000000
8,8,AMT_CREDIT,float64,5274,0.000000
9,9,AMT_ANNUITY,float64,13135,0.004065


In [20]:
bureau_cols = ['AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_DAY',
               'AMT_REQ_CREDIT_BUREAU_WEEK', 'AMT_REQ_CREDIT_BUREAU_MON',
               'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR']

df[bureau_cols].isna().value_counts()

AMT_REQ_CREDIT_BUREAU_HOUR  AMT_REQ_CREDIT_BUREAU_DAY  AMT_REQ_CREDIT_BUREAU_WEEK  AMT_REQ_CREDIT_BUREAU_MON  AMT_REQ_CREDIT_BUREAU_QRT  AMT_REQ_CREDIT_BUREAU_YEAR
False                       False                      False                       False                      False                      False                         212764
True                        True                       True                        True                       True                       True                           33244
Name: count, dtype: int64

In [21]:
missing = df['AMT_REQ_CREDIT_BUREAU_DAY'].isna()

# Categorical columns: share of each value in each group, side by side
cat_cols = df.select_dtypes(exclude='number').columns
for col in cat_cols:
    comp = pd.concat({
        'missing': df.loc[missing, col].value_counts(normalize=True),
        'present': df.loc[~missing, col].value_counts(normalize=True),
    }, axis=1).round(3)
    print(comp, '\n')

# Numeric columns: compare group means
num_cols = df.select_dtypes(include='number').columns
pd.concat({
    'missing': df.loc[missing, num_cols].mean(),
    'present': df.loc[~missing, num_cols].mean(),
}, axis=1)

                    missing  present
NAME_CONTRACT_TYPE                  
Cash loans            0.897    0.906
Revolving loans       0.103    0.094 

             missing  present
CODE_GENDER                  
F               0.63    0.663
M               0.37    0.337
XNA              NaN    0.000 

              missing  present
FLAG_OWN_CAR                  
N               0.709    0.653
Y               0.291    0.347 

                 missing  present
FLAG_OWN_REALTY                  
Y                  0.717     0.69
N                  0.283     0.31 

                 missing  present
NAME_TYPE_SUITE                  
Unaccompanied      0.796    0.814
Family             0.139    0.130
Spouse, partner    0.041    0.036
Children           0.012    0.011
Other_B            0.007    0.006
Other_A            0.004    0.003
Group of people    0.001    0.001 

                      missing  present
NAME_INCOME_TYPE                      
Working                 0.525    0.515
Commercia

,missing,present
SK_ID_CURR,278651.884881,278285.897445
TARGET,0.103387,0.077189
CNT_CHILDREN,0.380129,0.422078
AMT_INCOME_TOTAL,153034.136583,171324.878461
AMT_CREDIT,552655.166556,606632.398778
...,...,...
AMT_REQ_CREDIT_BUREAU_DAY,NaN,0.006895
AMT_REQ_CREDIT_BUREAU_WEEK,NaN,0.034245
AMT_REQ_CREDIT_BUREAU_MON,NaN,0.267103
AMT_REQ_CREDIT_BUREAU_QRT,NaN,0.264166


In [7]:
#unique categories in gender
df["CODE_GENDER"].value_counts()

CODE_GENDER
F      161887
M       84119
XNA         2
Name: count, dtype: int64

In [8]:
# percentage of applicants with AMT_GOODS_PRICE less than AMT_CREDIT with no missing values
df[df["AMT_GOODS_PRICE"]< df["AMT_CREDIT"]]

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,310536,0,Cash loans,F,N,N,2,90000.0,227520.0,13189.5,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0
1,365516,0,Cash loans,M,Y,Y,0,90000.0,161730.0,13095.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,242055,1,Cash loans,M,N,Y,0,135000.0,728847.0,26307.0,...,0,0,0,0,0.0,0.0,2.0,0.0,0.0,2.0
3,454894,1,Cash loans,M,N,N,0,135000.0,474183.0,34636.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,4.0
4,448321,0,Cash loans,F,N,Y,0,180000.0,254700.0,27558.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
246001,357427,1,Cash loans,F,N,Y,0,67500.0,352044.0,16411.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
246002,297241,0,Cash loans,F,N,Y,1,157500.0,846517.5,33700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,4.0
246004,240509,0,Cash loans,F,N,N,0,157500.0,272520.0,21528.0,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,4.0
246005,387513,0,Cash loans,F,N,N,0,90000.0,246357.0,24493.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0


In [9]:
#categories in occupation type
df['OCCUPATION_TYPE'].value_counts()

OCCUPATION_TYPE
Laborers                 44259
Sales staff              25606
Core staff               22093
Managers                 17096
Drivers                  14944
High skill tech staff     9093
Accountants               7882
Medicine staff            6871
Security staff            5372
Cooking staff             4767
Cleaning staff            3741
Private service staff     2094
Low-skill Laborers        1696
Secretaries               1047
Waiters/barmen staff      1046
Realty agents              594
HR staff                   443
IT staff                   424
Name: count, dtype: int64

In [10]:
#name_income_type
df["NAME_INCOME_TYPE"].value_counts()

NAME_INCOME_TYPE
Working                 127043
Commercial associate     57273
Pensioner                44134
State servant            17518
Unemployed                  17
Student                     11
Businessman                  9
Maternity leave              3
Name: count, dtype: int64

In [11]:
#now i want to check who are those people (their name_income_type) who don't have an occupation type
no_occ = df[df["OCCUPATION_TYPE"].isna()]
no_occ["NAME_INCOME_TYPE"].value_counts()

NAME_INCOME_TYPE
Pensioner               44130
Working                 19922
Commercial associate     9869
State servant            2997
Unemployed                 17
Businessman                 2
Student                     2
Maternity leave             1
Name: count, dtype: int64

In [12]:
#3 external sources- number of missing values in each
ext = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
df[ext].isna().sum()

EXT_SOURCE_1    138595
EXT_SOURCE_2       531
EXT_SOURCE_3     48805
dtype: int64

In [13]:
#apply describe on the three exteranla sources
df[ext].describe()

,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3
count,107413.000000,2.454770e+05,197203.000000
mean,0.501650,5.143034e-01,0.510871
std,0.211413,1.911257e-01,0.194771
min,0.014568,8.173617e-08,0.000527
25%,0.333105,3.921550e-01,0.370650
50%,0.505446,5.659131e-01,0.535276
75%,0.674781,6.635927e-01,0.669057
max,0.951624,8.549997e-01,0.896010


In [14]:
#correlation between the three external sources
df[ext].corr()

,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3
EXT_SOURCE_1,1.000000,0.214367,0.188942
EXT_SOURCE_2,0.214367,1.000000,0.109893
EXT_SOURCE_3,0.188942,0.109893,1.000000


In [15]:
#correlation of ext sources with the target
df[ext].corrwith(df["TARGET"])

EXT_SOURCE_1   -0.155362
EXT_SOURCE_2   -0.159593
EXT_SOURCE_3   -0.178845
dtype: float64

In [13]:
df[df['OWN_CAR_AGE'].isnull()]['FLAG_OWN_CAR'].value_counts()

FLAG_OWN_CAR
N    162413
Y         5
Name: count, dtype: int64

## Notes:

### Missing values:

- column description in HomeCredit_columns_description.csv
- number of applicants ~ 246k
- about 8% loan defaulters

- NAME_CONTRACT_TYPE, CODE_GENDER, FLAG_OWN_CAR, FLAG_OWN_REALTY - binary flag
- AMT_ANNUITY/AMT_INCOME_TOTAL can be a good feature
- AMT_GOODS_PRICE, AMT_ANNUITY- tiny amount of missing values can be handled by an imputer in the pipeline
- a lot of missing values in building related info- impute with median and add_indicator = True

- AMT_REQ_CREDIT_BUREAU_HOUR, AMT_REQ_CREDIT_BUREAU_WEEK,	AMT_REQ_CREDIT_BUREAU_MON,	AMT_REQ_CREDIT_BUREAU_YEAR,	AMT_REQ_CREDIT_BUREAU_DAY,	AMT_REQ_CREDIT_BUREAU_QRT- about 13% missing values- impute and add a flag

- OCCUPATION_TYPE: 31% missing. Pensioners ≈100% missing (explained by income type); ~33k working applicants missing (unexplained). → 'Missing' category.

- for missing values in EXT_SOURCE- impute with median and an indicator col
- EXT_SOURCE_1/2/3: corr with TARGET ≈ -0.16 to -0.18 (higher score → less default); weakly correlated with each other → keep all three separately, median + indicator

- NAME_TYPE_SUITE- missing

- DEF_60_CNT_SOCIAL_CIRCLE, OBS_60_CNT_SOCIAL_CIRCLE, DEF_30_CNT_SOCIAL_CIRCLE, OBS_30_CNT_SOCIAL_CIRCLE- 0.33% missing	- median impute

- CNT_FAM_MEMBERS 0.000813% missing- impute
- DAYS_LAST_PHONE_CHANGE 0.000406% missing - impute

would like to dig deeper if req (missing values <0.1%)
- AMT_GOODS_PRICE 0.089834% missing - impute
- AMT_ANNUITY 0.004065% missing - impute

In [16]:
with pd.option_context("display.max_rows", None):
    display(df.describe().T)

,count,mean,std,min,25%,50%,75%,max
SK_ID_CURR,246008.0,278335.354724,102797.868259,1.000020e+05,189239.750000,278392.000000,367291.750000,4.562550e+05
TARGET,246008.0,0.080729,0.272419,0.000000e+00,0.000000,0.000000,0.000000,1.000000e+00
CNT_CHILDREN,246008.0,0.416409,0.719500,0.000000e+00,0.000000,0.000000,1.000000,1.900000e+01
AMT_INCOME_TOTAL,246008.0,168853.180699,258960.632250,2.565000e+04,112500.000000,147600.000000,202500.000000,1.170000e+08
AMT_CREDIT,246008.0,599338.249368,402725.773305,4.500000e+04,270000.000000,514777.500000,808650.000000,4.050000e+06
AMT_ANNUITY,245998.0,27108.089708,14520.904087,1.615500e+03,16510.500000,24903.000000,34596.000000,2.580255e+05
AMT_GOODS_PRICE,245787.0,538652.980410,369595.049962,4.050000e+04,238500.000000,450000.000000,679500.000000,4.050000e+06
REGION_POPULATION_RELATIVE,246008.0,0.020858,0.013835,2.900000e-04,0.010006,0.018850,0.028663,7.250800e-02
DAYS_BIRTH,246008.0,-16029.515882,4362.869037,-2.522900e+04,-19673.000000,-15744.000000,-12401.000000,-7.489000e+03
DAYS_EMPLOYED,246008.0,63581.163637,141081.696923,-1.791200e+04,-2762.000000,-1215.000000,-291.000000,3.652430e+05


### Numerical columns
- CNT_CHILDREN max = 19 (need to check for outliers)
- DAYS_BIRTH has negative values - looks correct
- DAYS_EMPLOYED max = 1000 years (all pensioners) - replace with NaN and then median imputation and no indicator column needed (might also want to try 30-40 later)
- FLAG_MOBIL	Did client provide mobile phone (1=YES, 0=NO)	  
  FLAG_EMP_PHONE	Did client provide work phone (1=YES, 0=NO)	  
  FLAG_WORK_PHONE	Did client provide home phone (1=YES, 0=NO)	  
  FLAG_PHONE	Did client provide home phone (1=YES, 0=NO)	  
  
  these are weird descriptions

In [17]:
#DAYS_BIRTH in years
(df['DAYS_BIRTH']/-365).describe()

count    246008.000000
mean         43.916482
std          11.953066
min          20.517808
25%          33.975342
50%          43.134247
75%          53.898630
max          69.120548
Name: DAYS_BIRTH, dtype: float64

In [18]:
#same for days employed
(df['DAYS_EMPLOYED']/365).describe()

count    246008.000000
mean        174.194969
std         386.525197
min         -49.073973
25%          -7.567123
50%          -3.328767
75%          -0.797260
max        1000.665753
Name: DAYS_EMPLOYED, dtype: float64

In [19]:
df[(df['DAYS_EMPLOYED'])>0].sort_values('DAYS_EMPLOYED')['DAYS_EMPLOYED']

5         365243
24        365243
30        365243
36        365243
38        365243
           ...  
245985    365243
245988    365243
246005    365243
246006    365243
246007    365243
Name: DAYS_EMPLOYED, Length: 44143, dtype: int64

In [20]:
#no. of pension people
df[df['NAME_INCOME_TYPE'] =='Pensioner']

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
5,333032,0,Revolving loans,F,N,Y,0,74250.0,225000.0,11250.0,...,0,0,0,0,0.0,0.0,0.0,3.0,1.0,2.0
24,405722,0,Cash loans,F,N,Y,0,180000.0,669600.0,26685.0,...,0,0,0,0,0.0,0.0,0.0,0.0,2.0,3.0
30,355064,0,Cash loans,F,N,Y,0,90000.0,314100.0,15241.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
36,323806,0,Cash loans,F,N,Y,0,112500.0,523152.0,37336.5,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,5.0
38,321620,0,Cash loans,F,N,N,0,67500.0,490536.0,15003.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245985,130474,0,Cash loans,F,N,Y,0,112500.0,284400.0,16456.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
245988,127090,1,Cash loans,M,Y,Y,0,157500.0,840780.0,37165.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
246005,387513,0,Cash loans,F,N,N,0,90000.0,246357.0,24493.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
246006,303331,1,Cash loans,F,N,Y,0,112500.0,810000.0,26901.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,5.0


In [21]:
#what is DAYS_EMPLOYED for pensioner
df[df['NAME_INCOME_TYPE'] =='Pensioner']['DAYS_EMPLOYED'].value_counts()

DAYS_EMPLOYED
 365243    44126
-1669          1
-2341          1
-12865         1
-1346          1
-11194         1
-3261          1
-8140          1
-5111          1
Name: count, dtype: int64

In [22]:
df['OWN_CAR_AGE'].value_counts().sort_values()

OWN_CAR_AGE
57.0       1
47.0       1
50.0       1
52.0       1
56.0       1
        ... 
2.0     4683
8.0     4714
6.0     5058
3.0     5109
7.0     5940
Name: count, Length: 61, dtype: int64

In [10]:
df['CNT_CHILDREN'].value_counts()

CNT_CHILDREN
0     172232
1      49106
2      21304
3       2936
4        325
5         69
6         19
7          7
8          2
10         2
9          2
14         1
11         1
12         1
19         1
Name: count, dtype: int64

In [12]:
df[df['CNT_CHILDREN']>10].loc[:, ['CNT_CHILDREN', 'CNT_FAM_MEMBERS']]

,CNT_CHILDREN,CNT_FAM_MEMBERS
10505,14,16.0
82646,11,13.0
204061,12,14.0
231370,19,20.0


### Categorical column:
- CODE_GENDER Xna- missing
- ORGANIZATION_TYPE = XNa --> Pensioners 

In [23]:
#Loop over the str columns and print value_counts() for each- percentaage
str_cols = df.select_dtypes(include="str").columns

for col in str_cols:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False)/df.shape[0]*100)
    print()

--- NAME_CONTRACT_TYPE ---
NAME_CONTRACT_TYPE
Cash loans         90.468603
Revolving loans     9.531397
Name: count, dtype: float64

--- CODE_GENDER ---
CODE_GENDER
F      65.805584
M      34.193603
XNA     0.000813
Name: count, dtype: float64

--- FLAG_OWN_CAR ---
FLAG_OWN_CAR
N    66.019398
Y    33.980602
Name: count, dtype: float64

--- FLAG_OWN_REALTY ---
FLAG_OWN_REALTY
Y    69.383922
N    30.616078
Name: count, dtype: float64

--- NAME_TYPE_SUITE ---
NAME_TYPE_SUITE
Unaccompanied      80.813632
Family             13.040226
Spouse, partner     3.686059
Children            1.086550
Other_B             0.582095
NaN                 0.418279
Other_A             0.283324
Group of people     0.089834
Name: count, dtype: float64

--- NAME_INCOME_TYPE ---
NAME_INCOME_TYPE
Working                 51.641817
Commercial associate    23.280950
Pensioner               17.940067
State servant            7.120907
Unemployed               0.006910
Student                  0.004471
Businessman     

In [24]:
df[df['ORGANIZATION_TYPE']== 'XNA']['NAME_INCOME_TYPE'].value_counts()

NAME_INCOME_TYPE
Pensioner     44126
Unemployed       17
Name: count, dtype: int64

## Decison list
SK_ID_CURR - drop    
NAME_CONTRACT_TYPE - one hot encoding (2 categories - Cash loans, Revolving loans)         
CODE_GENDER - one hot encoding (3 categories- M, F, XNA)        
FLAG_OWN_CAR - one hot encoding (2 categories - Y, N)      FLAG_OWN_REALTY - one hot encoding (2 categories - Y, N)         
CNT_CHILDREN - no change needed       
AMT_INCOME_TOTAL     
AMT_CREDIT        
AMT_ANNUITY - missing values (imputer)        
AMT_GOODS_PRICE- missing values (imputer)       
NAME_TYPE_SUITE - missing values (replace the NaN with Unaccompanied - the mode)     
NAME_INCOME_TYPE - one hot encoding (8 categories)       
NAME_EDUCATION_TYPE - ordinal encoding (order = Lower secondary < Secondary / secondary special < Incomplete higher < Higher education < Academic degree)       
NAME_FAMILY_STATUS - one hot encoding (6 categories)       
NAME_HOUSING_TYPE - one hot encoding (6 categories)        
REGION_POPULATION_RELATIVE        
DAYS_BIRTH      
DAYS_EMPLOYED - replace 365243 with NaN- then impute with median      
DAYS_REGISTRATION        
DAYS_ID_PUBLISH                      
OWN_CAR_AGE - missing values - impute with median        
FLAG_MOBIL         
FLAG_EMP_PHONE         
FLAG_WORK_PHONE        
FLAG_CONT_MOBILE              
FLAG_PHONE 
FLAG_EMAIL          
OCCUPATION_TYPE - missing values- fill with "missing" - one hot encoding         
CNT_FAM_MEMBERS - missing values- median imputation        
REGION_RATING_CLIENT        
REGION_RATING_CLIENT_W_CITY         
WEEKDAY_APPR_PROCESS_START- one hot encoding (7 categories)       
HOUR_APPR_PROCESS_START       
REG_REGION_NOT_LIVE_REGION        
REG_REGION_NOT_WORK_REGION         
LIVE_REGION_NOT_WORK_REGION             
REG_CITY_NOT_LIVE_CITY          
REG_CITY_NOT_WORK_CITY              
LIVE_CITY_NOT_WORK_CITY              
ORGANIZATION_TYPE - one hot encoding (a lots of categories)        
EXT_SOURCE_1 - missing values- median imputation with an indicator   
EXT_SOURCE_2 - missing values- median imputation     
EXT_SOURCE_3 - missing values- median imputation with an indicator    

------------- missing values - median imputation with an indicator------------      
APARTMENTS_AVG - missing values - median imputation with an indicator  
BASEMENTAREA_AVG - missing values - median imputation with an indicator   
YEARS_BEGINEXPLUATATION_AVG - missing values - median imputation with an indicator    
YEARS_BUILD_AVG - missing values        
COMMONAREA_AVG- missing values    
ELEVATORS_AVG- missing values       
ENTRANCES_AVG- missing values      
FLOORSMAX_AVG- missing values            
FLOORSMIN_AVG- missing values        
LANDAREA_AVG- missing values       
LIVINGAPARTMENTS_AVG- missing values      
LIVINGAREA_AVG- missing values         
NONLIVINGAPARTMENTS_AVG- missing values      
NONLIVINGAREA_AVG- missing values        
APARTMENTS_MODE- missing values          
BASEMENTAREA_MODE- missing values           
YEARS_BEGINEXPLUATATION_MODE- missing values        
YEARS_BUILD_MODE- missing values        
COMMONAREA_MODE- missing values            
ELEVATORS_MODE- missing values             
ENTRANCES_MODE- missing values           
FLOORSMAX_MODE- missing values           
FLOORSMIN_MODE- missing values             
LANDAREA_MODE- missing values         
LIVINGAPARTMENTS_MODE- missing values  
LIVINGAREA_MODE- missing values         
NONLIVINGAPARTMENTS_MODE- missing values    
NONLIVINGAREA_MODE- missing values     
APARTMENTS_MEDI- missing values     
BASEMENTAREA_MEDI- missing values           
YEARS_BEGINEXPLUATATION_MEDI- missing values    
YEARS_BUILD_MEDI- missing values      
COMMONAREA_MEDI- missing values         
ELEVATORS_MEDI- missing values      
ENTRANCES_MEDI- missing values          
FLOORSMAX_MEDI- missing values             
FLOORSMIN_MEDI- missing values           
LANDAREA_MEDI- missing values           
LIVINGAPARTMENTS_MEDI- missing values        
LIVINGAREA_MEDI- missing values      
NONLIVINGAPARTMENTS_MEDI- missing values   
NONLIVINGAREA_MEDI- missing values     
- -------------missing values - median imputation with an indicator----------------                                
FONDKAPREMONT_MODE- missing values- impute with "missing"   
HOUSETYPE_MODE- missing values- impute with "missing"    
TOTALAREA_MODE- missing values- median imputation with an indicator   
WALLSMATERIAL_MODE- missing values- impute with "missing"     
EMERGENCYSTATE_MODE- missing values - impute with "missing"   

OBS_30_CNT_SOCIAL_CIRCLE- missing values - median imputation   
DEF_30_CNT_SOCIAL_CIRCLE- missing values - median imputation    
OBS_60_CNT_SOCIAL_CIRCLE- missing values - median imputation     
DEF_60_CNT_SOCIAL_CIRCLE- missing values - median imputation    
DAYS_LAST_PHONE_CHANGE- missing values - median imputation   
FLAG_DOCUMENT_2    
FLAG_DOCUMENT_3    
FLAG_DOCUMENT_4    
FLAG_DOCUMENT_5    
FLAG_DOCUMENT_6    
FLAG_DOCUMENT_7        
FLAG_DOCUMENT_8    
FLAG_DOCUMENT_9    
FLAG_DOCUMENT_10    
FLAG_DOCUMENT_11    
FLAG_DOCUMENT_12    
FLAG_DOCUMENT_13    
FLAG_DOCUMENT_14    
FLAG_DOCUMENT_15    
FLAG_DOCUMENT_16    
FLAG_DOCUMENT_17    
FLAG_DOCUMENT_18    
FLAG_DOCUMENT_19        
FLAG_DOCUMENT_20    
FLAG_DOCUMENT_21    
------median imputation and only one indicator for all----------------    
AMT_REQ_CREDIT_BUREAU_HOUR- missing values    
AMT_REQ_CREDIT_BUREAU_DAY- missing values    
AMT_REQ_CREDIT_BUREAU_WEEK- missing values    
AMT_REQ_CREDIT_BUREAU_MON- missing values    
AMT_REQ_CREDIT_BUREAU_QRT- missing values    
AMT_REQ_CREDIT_BUREAU_YEAR- missing values
